# PEMS Capstone Part 1 — Check: Navigation and Actions
## Instructor Answer Key

**Companion to:** `Capstone_Part1_Check.ipynb` (the attached student notebook). This key follows **the student's Sections 1–5 in the same order**; it is not the older simulated A–E answer notebook.

> **Important:** Several prompts require a student to predict and observe the behavior of **their own** `PEMS/`. Consequently, there is no universal screen transcript or stack trace. The answers below provide a concrete worked example and the logic to check. Grade what the student's actual code supports, not whether their chosen names, amounts, or menu numbers match the example.

**Assumptions used in the worked example**

- `client_management` is the first menu; its options are Select Client (1) and Create Client (2).
- `client_workspace` provides Transactions (1) and Reports (2).
- Enter pops the menu stack. Successful Select Client pushes `client_workspace`.
- The wrapper treats **only the actual boolean `False`** returned by an action as a reason to stay in the current menu. An implicit `None` is not `False`.
- Create Client has no `next_menu`; it adds a client and an opening cash contribution.
- In Part 1, the sample option chosen under Transactions does not alter data.

If a student's menu data or wrapper differs, adjust the sample trace accordingly. This answer key **does not change project files or make Git commits**.


---
## Setup

The following helper functions are retained from the student notebook to make comparison easy. They are **optional** for reading this key. If a `PEMS/` folder is not present in the kernel's working directory, they report that fact without crashing. The answers do not depend on running them.


In [1]:
import ast
import json
from pathlib import Path

PEMS_DIR = Path("PEMS")
DATA = PEMS_DIR / "data"
SRC = PEMS_DIR / "source_code"

if not (DATA / "pems.json").exists() or not (SRC / "pems.py").exists():
    print("No project found at ./PEMS/ — normal when reading this instructor key separately.")
else:
    print("Found PEMS at", PEMS_DIR.resolve())


def show_pems(pm=None):
    """Show persisted clients and their transactions in the local project, if present."""
    path = DATA / "pems.json"
    if not path.is_file():
        print("Cannot display data: PEMS/data/pems.json is not available here.")
        return
    with path.open(encoding="utf-8") as file:
        pems = json.load(file)
    for initials, record in pems.items():
        if pm is not None and initials != pm:
            continue
        clients = record["clients"]
        print(f"{initials}: {len(clients)} client(s)")
        for idx, client in enumerate(clients):
            print(f"  [{idx}] {client.get('name')}  ({len(client.get('transactions', []))} transaction(s))")
            for t in client.get("transactions", []):
                total = t.get("tot_cst", t.get("total"))
                print(f"        #{t.get('trans_id')} day {t.get('trd_day')} {t.get('trans_type')} "
                      f"{t.get('ticker')} total={total}")


def show_menus():
    """Display menu options in the local project, if present."""
    path = DATA / "menus.json"
    if not path.is_file():
        print("Cannot display menus: PEMS/data/menus.json is not available here.")
        return
    with path.open(encoding="utf-8") as file:
        menus = json.load(file)
    for name, menu in menus.items():
        print(f"{name}  ({menu['title']})")
        for number, option in enumerate(menu["options"], 1):
            print(f"  {number}) {option['menu_item']:16} next_menu={option.get('next_menu')!r:20} "
                  f"action={option.get('action', '<no action key>')!r}")


def show_function(name, filename="actions.py"):
    """Display a function from the local project, if present."""
    path = SRC / filename
    if not path.is_file():
        print(f"Cannot display {name}: {path} is not available here.")
        return
    source = path.read_text(encoding="utf-8")
    for node in ast.parse(source).body:
        if isinstance(node, ast.FunctionDef) and node.name == name:
            print(ast.get_source_segment(source, node))
            return
    print(f"{name} not found in {filename}")


Found PEMS at C:\PythonClass\class_repo\labs\capstone\part_1\PEMS


In [2]:
show_menus()

client_management  (CLIENT MANAGEMENT)
  1) Select Client    next_menu='client_portfolio'   action='select_client'
  2) Create Client    next_menu=None                 action='create_client'
client_portfolio  (CLIENT PORTFOLIO)
  1) Transactions     next_menu='transactions'       action=None
  2) Reports          next_menu='reports'            action=None
transactions  (TRANSACTIONS)
  1) Buy Stock        next_menu=None                 action=None
  2) Sell Stock       next_menu=None                 action=None
  3) Contribution     next_menu=None                 action=None
  4) Withdrawal       next_menu=None                 action=None
  5) Advance Trade Day next_menu=None                 action=None
reports  (REPORTS)
  1) Holdings         next_menu=None                 action=None
  2) Transactions     next_menu=None                 action=None
  3) Performance      next_menu=None                 action=None
  4) Cash Summary     next_menu=None                 action=None


---
## 1 · Trace your own system

You'll run PEMS along a route you plan, but you'll predict every step first.

**Start clean.** Run `create_pm.py` and make a new PM just for this check (pick your own initials). A new PM has no clients, which you need for step 1.

**The route.** Choose your own client names and cash amounts. Your route must include these steps, in this order:

1. Select Client (with no clients yet)
2. Create Client with one invalid entry (your choice of what's wrong)
3. Create Client, valid
4. Create Client, valid (a second client)
5. Select Client, and choose the **second** client
6. Transactions, then one option, then Enter
7. Reports, then Enter
8. Enter until PEMS logs you off

### 1a · Predict

**Before running anything**, fill in the cell below. Write one line for every key you press, including each Enter, with these columns separated by `|`:

step | menu on screen | you enter | stack after | `client_idx` after | change to `pems.json` (or "nothing")

The first line is started for you.

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### 1a · Sample predicted route

Let `CM = client_management`, `CW = client_workspace`, `T = transactions`, and `R = reports`. Example entries: initial invalid amount `abc`; valid clients **Ada ($1,000)** and **Grace ($500)**. A row describes one complete *menu choice* (including any subsequent answers at that option's prompts); this is not a character-by-character log.

| Step | Menu on screen | Input/action | Stack after | `client_idx` after | Change to `pems.json` |
|---:|---|---|---|---|---|
| 1 | Client Management | `1` Select Client with no clients | `[CM]` | `None` | None; action returns `False` |
| 2 | Client Management | `2` Create Client, invalid starting cash `abc` | `[CM]` | `None` | None; invalid input |
| 3 | Client Management | `2` Create Ada, $1,000 | `[CM]` | `None` | Ada appended; her opening contribution recorded |
| 4 | Client Management | `2` Create Grace, $500 | `[CM]` | `None` | Grace appended; her opening contribution recorded |
| 5 | Client Management | `1` Select Client, then choose `2` from list | `[CM, CW]` | `1` | None |
| 6 | Client Workspace | `1` Transactions | `[CM, CW, T]` | `1` | None |
| 7 | Transactions | One option with no implemented action/next menu | `[CM, CW, T]` | `1` | None |
| 8 | Transactions | Enter | `[CM, CW]` | `1` | None |
| 9 | Client Workspace | `2` Reports | `[CM, CW, R]` | `1` | None |
| 10 | Reports | Enter | `[CM, CW]` | `1` | None |
| 11 | Client Workspace | Enter | `[CM]` | `1`, then typically cleared to `None` when CM displays | None |
| 12 | Client Management | Enter | `[]` | `None` in the example implementation | None; log off |

**Checks on the table:** Python list indices start at zero, so selecting displayed client **2** stores `client_idx = 1`. After the two valid creations, `show_pems()` should show **two clients**, each with **one opening CONTRIBUTION**; no invalid client should have been written.

**Not universal:** Different option numbering, a different transaction action, menu validation rules, or exactly when `client_idx` is cleared can change these rows. A student's trace should reflect their actual menu options, action return values, and loop order.

### 1b · Run it

Run `pems.py` in the terminal and follow your route exactly. Then run the next cell to see what your `pems.json` holds now.

In [3]:
show_pems()   # or show_pems("XYZ") with your new PM's initials

RSS: 0 client(s)


### 1c · Reconcile

- Which predictions were wrong? For each one, what actually happened, and **which line of code** explains it? Name the file and the function.
- If everything matched, pick the step you were least sure of and explain which code in pems.py and or actions.py decides its outcome.
- Your stack column: `pems.py` never prints the stack. How did you know what it was?

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### 1c · Example reconciliation and reasoning

**Example correction:** "I initially thought choosing Create Client would push Client Workspace. It did not. The `Create Client` option in `data/menus.json` has `next_menu: null`, so `pems.py` had no submenu to push, even though `create_client` succeeded."

**Other important explanations to look for:**

- Selecting a client when no clients exist returns `False` from `select_client` in `source_code/actions.py`; the wrapper should not push `client_workspace`.
- A valid `create_client` builds a client dictionary, appends the opening transaction, calls `clients.append(client)`, and then `save_pems(state)`. Invalid input returns `False` before those appends.
- `select_client` stores the **zero-based** index `choice - 1` in `state['client_idx']`.
- The stack is *inferred from the `pems.py` algorithm*: start with the initial menu, push `next_menu` only when the option proceeds, and pop once for each Enter. The wrapper need not print the stack for the student to predict it.

A fully correct answer identifies a relevant function or decision in the **student's actual files** and compares the prediction with an observed run. "Everything matched" is acceptable only with a specific explanation of one uncertain step. The key cannot supply an observed mismatch for a student who has not run PEMS.

---
## 2 · Break it on purpose

Each experiment makes one small change to your code. For each one:

1. **Predict**, before you change anything: what will you see, and when? At startup, at login, when you choose an option, or never?
2. Make the change, run `pems.py`, and **describe what you saw**.
3. **Explain** why, in terms of names, binding, the stack, return values, or references.
4. **Restore** the file with `git restore` (for example `git restore "PEMS/source code/actions.py"`) and check that PEMS works again.

**First, run the next cell.** It commits your PEMS folder as it is now, so `git restore` brings back *your* finished code, not an older version. Run it once, before Experiment A.

**Instructor-key safety note:** The next cell is **disabled** here. The student notebook runs a Git commit to establish a baseline; reading an answer key should not commit or restore another person's files. When interpreting the student's observations, check that they restored each experiment before the next one.

In [ ]:
# Instructor key: no Git write operations.
print("The student notebook's Git snapshot step is intentionally disabled in this answer key.")
print("From part_1: git restore PEMS/source_code/actions.py")
print("From inside PEMS/: git restore source_code/actions.py")


### Experiment A · An unregistered action

In `actions.py`, delete the `"create_client": create_client` line from the `actions` dictionary.

Hint for your prediction: when does `load_data()` run, relative to `login()`?

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### Experiment A · Unregistered action

**Prediction:** In the expected implementation, removing the mapping for `create_client` causes an error **while loading/binding the menus**, often at application startup, before a user can choose Create Client or log in. The error is typically `KeyError: 'create_client'` (or a deliberately generated diagnostic).

**Expected observation:** The project fails when `load_data()` resolves the `"create_client"` string in `menus.json` against the `actions` dictionary. Exact timing and error display depend on whether `load_data()` runs before or after login and whether the code catches the exception.

**Why:** A menu's action starts as a string name. If the binding step does something like `actions[action_name]`, a missing key cannot be replaced by a callable function. The failure is **name resolution/binding**, not execution of `create_client`.

**Grade for:** Explicit discussion of action names versus function references; when `load_data()` executes; and restoration of the deleted mapping before Experiment B.

### Experiment B · A misspelled menu name

In `data/menus.json`, change Client Workspace's Reports option from `"next_menu": "reports"` to `"next_menu": "report"`.

Compare with Experiment A: both are a misspelled name. Do they fail at the same moment? Why or why not?

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### Experiment B · Misspelled `next_menu`

**Prediction:** Changing `"reports"` to `"report"` leaves a string that does **not** name any menu key.

**Expected behavior if there is no proactive menu validator:** PEMS loads, the user can log in and navigate to Client Workspace, and choosing Reports **pushes the string `report` onto the stack**. On the next loop iteration, looking up `menus["report"]` raises `KeyError: 'report'`.

**Possible alternate behavior:** If the student's `load_data()` or startup validator checks **all** `next_menu` targets, it can detect the invalid name **at load/startup** instead. That is also a correct result if backed by the code.

**Comparison with A:** A missing action name usually fails when the system **binds actions**. An invalid `next_menu` name can remain just a string until navigation uses it, **unless** a validator checks menu links earlier. The two failures have different lookup points.

### Experiment C · Cancel without `False`

In `select_client`, change the cancel line `return False` (the one after `if entry == '':`) to just `return`. Then run PEMS, choose Select Client, and press Enter at the client list.

What does the user see? What is `state["client_idx"]` now? What would go wrong later, when Contribution is a real action?

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### Experiment C · Cancel without `False`

**Prediction:** At the client list, pressing Enter executes a bare `return`, whose value is `None`. The `client_idx` remains `None` because `select_client` cleared it at entry and no client was chosen.

**Expected observation with the course wrapper (`result is False`):** `None is False` evaluates to `False`, so the wrapper treats the action as successful and follows Select Client's `next_menu`, displaying **Client Workspace with no selected client**.

**Why it matters later:** Contribution or Withdrawal may do something like `state['clients'][state['client_idx']]`. Here the index is `None`; indexing a list with `None` raises `TypeError`. Even if the action checks the index safely, the user should never enter that workspace without a valid selection.

**Important nuance:** If the student's wrapper instead tests `if not result`, then `None` is falsey and the cancellation will stay on the menu. The student must explain the observed behavior using the **actual** wrapper condition, rather than guessing.

### Experiment D · A new list instead of the same list

In `create_client`, replace `clients.append(client)` with:

```python
state["clients"] = state["clients"] + [client]
```

Then run PEMS and:

1. Create a client. Before going on, run `show_pems()` in this notebook to see what was saved.
2. Choose Select Client. Is your new client listed? Choose it.

Predict both steps before you run them. Then explain the result in terms of the two lists involved: `state["clients"]`, and `state["pems"][pm]["clients"]`, which `pems.py` and `save_pems` use.

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### Experiment D · Replace append with list concatenation and assignment

**Prediction for step 1:** `create_client` appears successful, but `show_pems()` (which reads persisted `pems.json`) **does not show the newly created client**, assuming `save_pems` writes `state['pems']`.

**Prediction for step 2:** In the **same running PEMS session**, Select Client **does show the new client**, because `select_client` reads the new list now assigned to `state['clients']`. The user can select it. After restarting/reloading from disk, that unsaved client is gone.

**Why:** Initially, `state['clients']` and `state['pems'][pm]['clients']` refer to the **same list**. This operation creates a *new* list and reassigns only one dictionary entry:

```python
state['clients'] = state['clients'] + [client]
```

The original list inside `state['pems'][pm]['clients']` remains unchanged; it is the list that `save_pems` serializes in the specified design. By contrast:

```python
clients.append(client)
```

mutates the existing shared list *in place*, preserving both references to it.

**Caveat:** If a student's `save_pems` implementation serializes `state['clients']` directly or rewrites `state['pems'][pm]['clients']`, they may observe different results. The answer must cite their actual save path.

In [ ]:
# Optional instructor demonstration of Experiment D — no project files touched.
pems_demo = {"ABC": {"clients": []}}
state_demo = {"pm": "ABC", "pems": pems_demo,
              "clients": pems_demo["ABC"]["clients"]}
print("Initially same list:", state_demo["clients"] is pems_demo["ABC"]["clients"])
state_demo["clients"] = state_demo["clients"] + [{"name": "New client"}]
print("After reassignment, same list:", state_demo["clients"] is pems_demo["ABC"]["clients"])
print("Visible to select_client:", state_demo["clients"])
print("Visible in persisted pems structure:", pems_demo["ABC"]["clients"])


**Discuss with the tutor:** pick the experiment whose result surprised you most, or that you predicted least confidently. Explain your reasoning to the tutor and let it question you. Then record:

- What the tutor asked that you couldn't answer right away.
- Whether your explanation changed, and how.

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### Tutor discussion · Example of an acceptable reflective entry

**This answer cannot be predetermined**—the student must discuss a real experiment with the tutor. A *sample* reflective entry might be:

> "The tutor asked whether `state['clients'] = state['clients'] + [client]` modifies the old list or constructs a new one. I had assumed both lists would be updated. I checked the code and realized that concatenation creates a new list, so the persistent `pems` dictionary still holds the old list. I revised my explanation of why selection worked but saving did not."

Look for a **specific tutor question**, the student's **initial reasoning**, and a genuine **revision or defensible non-revision**, not simply "the tutor explained it."

---
## 3 · Comment your own `create_client`

Add comments to **your** `create_client` in `actions.py`: the code you wrote for the two TODOs, and the two FATAL ERROR guards. Comments are your own work; the tutor won't write them for you.

Good comments say **why**, not what. `# append the client` adds nothing. `# append, don't rebind: state["clients"] is the same list save_pems writes` shows understanding.

When you're done, run the cell to include your function in this notebook.

**Instructor guidance:** Assess the student's own comments; the snippets below illustrate the *quality and purpose* of comments, not a required implementation.

In [ ]:
show_function("create_client")

**One question about your code:** `create_client` builds the transaction and appends it to `client["transactions"]` **before** appending the client to `clients`. Would reversing those two appends change what's saved? Why or why not?

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### Section 3 · Annotated `create_client` reasoning and append order

**Examples of useful comments to look for in the student's function:**

```python
# Reject incomplete client data before changing any persisted lists.
if not client_created:
    return False

# A new client must start with its own empty transaction history.
client = {'name': name, 'email': email, 'phone': phone,
          'transactions': []}

# Fail immediately if our client record violates the agreed schema;
# continuing would save malformed data and break other actions.
if set(client) != set(CLIENT_KEYS):
    print('FATAL ERROR: Client record incomplete')
    sys.exit()

# Cash is represented as a contribution transaction, not as a
# separate stored balance; later holdings can be derived from history.
transaction = { ... }   # all ten TRANSACTION_KEYS in real code

# The transaction must satisfy the shared schema before we save it.
if set(transaction) != set(TRANSACTION_KEYS):
    print('FATAL ERROR: Transaction record incomplete')
    sys.exit()

# Append to the existing list shared by state and the persistent pems dict.
transactions.append(transaction)
clients.append(client)
```

**The append-order question:** Reversing `transactions.append(transaction)` and `clients.append(client)` would **not change what gets saved**, **if both occur before `save_pems(state)` and no intervening code observes or saves a partial client**. The client dictionary is the **same mutable object** in both places. Adding it to `clients` does not make a detached copy; appending to `client['transactions']` later still changes that object.

That does **not** mean the order is meaningless for readability or safety. Recording the opening contribution **before** publishing/saving the client makes it less likely another code path will see a temporarily incomplete client.

---
## 4 · Design Contribution and Withdrawal

The next assignment makes Contribution and Withdrawal real actions. **Design them now; don't code them.** Use the ISTI model, and write this section before you discuss it with the tutor.

### 4a · Your design

For **each** of `record_contribution` and `record_withdrawal`:

- **Inputs:** what does the user enter, and what makes an entry invalid?
- **State:** which parts of `state` does the action read, and which does it change?
- **Transitions:** what happens on success? On each kind of failure?
- **Invariants:** what must always be true? (For example, think about how much a client can withdraw.)
- **Return value:** what does the action return in each outcome, and what does the user see next?
- **The transaction:** write the complete dictionary for one example, with all ten `TRANSACTION_KEYS`.
- **`menus.json`:** the option entries you'd add, with `menu_item`, `next_menu`, and `action`.

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### 4a · Example ISTI design (no function implementation required)

#### `record_contribution(state)`

- **Inputs:** An amount typed by the user. Allow cancellation on Enter if the UI specifies it. Reject nonnumeric input, blank when not intended as cancellation, `nan`, `inf`, negative or zero amounts, and values rounding to zero cents.
- **State read:** `state['clients']`, `state['client_idx']`, `state['trd_day']`; the selected client's existing transaction list.
- **State changed:** Append **one** CONTRIBUTION to the selected client's `transactions`; persist through `save_pems(state)`. Do not create a separate stored cash balance.
- **Transitions:** On success, create the transaction, append, save, show confirmation. On cancellation/invalid input/no selected client, do not append or save a transaction.
- **Invariants:** Selected client exists; amount is a finite positive number with two-decimal precision; all ten transaction keys are present; next `trans_id` is unique/sequential within that client.
- **Return/menu behavior:** Return `False` on cancellation or failure to block navigation; an implicit `None` or `True` can mean success if the wrapper tests `is False`. With `next_menu: null`, remain in Transactions on either outcome.

**Complete example CONTRIBUTION transaction**, assuming the client already has transaction #1 and the trading day is 1:

```python
{
    'trans_id': 2,
    'trd_day': 1,
    'trans_type': 'CONTRIBUTION',
    'ticker': '$$$$',
    'shares': 200.00,
    'cps': 1.00,
    'tot_cst': 200.00,
    'div_amnt': None,
    'shares_before': 1,
    'shares_after': 1
}
```

#### `record_withdrawal(state)`

- **Inputs:** An amount. Apply the same numeric validation. Reject a withdrawal **exceeding available cash**.
- **State read:** Selected client and their transaction history to **derive available cash**; trading day and next transaction ID. Under the Part 1 cash-only simplification, cash is contributions minus withdrawals. When Buy/Sell is implemented, their cash effects must be included.
- **State changed:** Append one WITHDRAWAL transaction and save only on success; no separate cash field needs updating.
- **Transitions:** On insufficient funds or invalid input, explain and do nothing. On success, record the withdrawal and confirm it.
- **Invariants:** Derived available cash must not become negative; transaction schema and identifiers remain valid; withdrawal amount is positive, even though it **reduces** cash because of its `trans_type`.
- **Return/menu behavior:** Same `False`-on-failure policy as Contribution.

**Complete example WITHDRAWAL transaction**, assuming transaction #2 was the $200 contribution above and there is at least $50 cash remaining:

```python
{
    'trans_id': 3,
    'trd_day': 1,
    'trans_type': 'WITHDRAWAL',
    'ticker': '$$$$',
    'shares': 50.00,
    'cps': 1.00,
    'tot_cst': 50.00,
    'div_amnt': None,
    'shares_before': 1,
    'shares_after': 1
}
```

**Example options to add in `data/menus.json`:**

```json
{"menu_item": "Contribution", "next_menu": null, "action": "record_contribution"}
{"menu_item": "Withdrawal",   "next_menu": null, "action": "record_withdrawal"}
```

Also register each name in the `actions` dictionary so `load_data()` can bind it. The provided **student question** asks only for a design, not working action functions. Use the current transaction names (`cps`, `tot_cst`, and `trans_type`), not older aliases (`cost_per_share`, `total`, or `trn_type`).

### 4b · Pseudocode

Plain English steps, not Python, for `record_withdrawal`.

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### 4b · Pseudocode for `record_withdrawal`

1. Check that a valid `client_idx` identifies an existing client. If not, explain the problem and report failure.
2. Retrieve the client's transactions from `state['clients']`.
3. Derive available cash from the transaction history. For Part 1's cash-only model, add contributions and subtract withdrawals; later include Buy/Sell cash flows when those are implemented.
4. Ask for the withdrawal amount. If the user cancels, stop without changing state.
5. Convert the input to a number, round to cents, and verify it is finite and greater than zero. If not, print an error and stop without changing state.
6. If the requested amount is more than the client's available cash, print "Insufficient cash" and stop without changing state.
7. Build a WITHDRAWAL transaction using all ten required fields; set its new sequential ID and current trading day.
8. Append the transaction to the **existing** selected-client transaction list.
9. Save PEMS, show a success message, and return success (or allow an implicit `None` if that is the established convention).

**Key design idea:** No cash balance is stored as a second source of truth. The transaction list is canonical, and cash is derived from it.

### 4c · Discuss with the tutor

Take your design and pseudocode to the tutor. Then record:

- One challenge or edge case the tutor raised.
- Whether you changed your design because of it. If you did, what changed? If you didn't, why not?

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### 4c · Tutor discussion · Example, not a predetermined result

A *plausible* student response is:

> "The tutor asked what happens if a user requests a withdrawal of exactly the available cash. I decided that should be permitted (cash becomes zero) and clarified that the invalid case is *strictly greater than* available cash. The tutor also asked whether Buy/Sell should affect available cash; I added that dependency to the design for Part 2."

**Assessment:** The student should state a **specific edge case**, explain whether it changed the design, and connect the decision to an invariant. This section documents an actual discussion; copying the sample would not meet that goal.

---
## 5 · Reflection

In a few sentences: what is one thing about navigation or actions you had wrong, or only half right, before this check? How did you find out?

**Instructor answer — illustrative (adapt to the actual student PEMS)**

### 5 · Reflection · Example

> "Before this check I thought `state['clients'] = state['clients'] + [client]` was effectively the same as `clients.append(client)`. In Experiment D I saw that the new client could appear in Select Client but did not appear in `show_pems()`. I traced the difference: concatenation made a new list and updated only one reference, while `append` changes the list shared by `state['clients']` and `state['pems'][pm]['clients']`."

Other sound reflections could concern the difference between a function **name** and a function **reference**, why `False` differs from an implicit `None`, or the distinction between the **menu stack** and `client_idx`. The important evidence is a specific prior misconception and a concrete way it was resolved.

---
## Instructor review checklist

- [ ] Section 1 includes a **prediction before running**, the menu/stack/`client_idx` trace, and an explanation of what `show_pems()` confirms.
- [ ] Section 2 covers **all four experiments** and distinguishes predicted results from actual observations; modified files were restored after every experiment.
- [ ] The student understands **function-name binding**, `next_menu` lookup, `False` versus `None`, and **aliasing versus rebinding**.
- [ ] Section 3 comments explain *why* decisions are made and correctly explain the two appends.
- [ ] Section 4 uses **ISTI** (Inputs, State, Transitions, Invariants) and supplies two **complete ten-key transactions**, a `menus.json` entry for each action, and withdrawal pseudocode.
- [ ] Sections 2 and 4 record genuine discussions with the tutor; Section 5 names a concrete learning or corrected misconception.
- [ ] The project's actual action names, menu keys, schema, and `save_pems` logic are used where they differ from the illustrative key.

**Answer key scope:** Derived from `Capstone_Part1_Check(1).ipynb` and the code conventions shown in this conversation. A student's own project files are needed to verify implementation-specific screen behavior. This key deliberately avoids Git commits, `git restore`, or modification of `PEMS/`.
